## Non negative factor decomposition on NIFTY 50 index and its constituents ##


In [1]:

import pandas as pd
import yfinance as yf
from sklearn.decomposition import NMF

In [18]:

# Full list of NIFTY 50 tickers on Yahoo Finance (suffix .NS for NSE stocks)
tickers = [
    "^NSEI",  # NIFTY 50 index itself
    "ADANIPORTS.NS", "ASIANPAINT.NS", "AXISBANK.NS", "BAJAJ-AUTO.NS", "BAJFINANCE.NS",
    "BAJAJFINSV.NS", "BHARTIARTL.NS", "BPCL.NS", "BRITANNIA.NS", "CIPLA.NS",
    "COALINDIA.NS", "DIVISLAB.NS", "DRREDDY.NS", "EICHERMOT.NS", "GRASIM.NS",
    "HCLTECH.NS", "HDFCBANK.NS", "HDFCLIFE.NS", "HEROMOTOCO.NS", "HINDALCO.NS",
    "HINDUNILVR.NS", "ICICIBANK.NS", "INDUSINDBK.NS", "INFY.NS", "ITC.NS",
    "JSWSTEEL.NS", "KOTAKBANK.NS", "LT.NS", "M&M.NS", "MARUTI.NS",
    "NESTLEIND.NS", "NTPC.NS", "ONGC.NS", "POWERGRID.NS", "RELIANCE.NS",
    "SBILIFE.NS", "SBIN.NS", "SUNPHARMA.NS", "TATACONSUM.NS", 
    "TATASTEEL.NS", "TCS.NS", "TECHM.NS", "TITAN.NS", "ULTRACEMCO.NS",
    "UPL.NS", "WIPRO.NS"
]

# Example: download last 3 years of daily closing prices
data = yf.download(tickers, start="2023-04-01", end="2026-04-01")["Close"]

# Inspect first few rows
# print(data.head())

data.to_pickle("nifty50_closing_prices.pkl")
data=pd.read_pickle("nifty50_closing_prices.pkl")


[*********************100%***********************]  47 of 47 completed


In [19]:
data=data.pct_change()
data.dropna(inplace=True)
corr_ = data.corr() 
modified_corr=corr_ + 1 #This is used to shift all the values by 1, resulting into non negative numbers 

In [53]:
n_components = 8  # Number of factors to extract
model = NMF(n_components=n_components, init='random', random_state=0,max_iter=20000)
W = model.fit_transform(modified_corr)  # Factor loadings
H = model.components_  # Factor scores
W_df = pd.DataFrame(W, index=data.columns)
W_df=W_df.drop(index=["^NSEI"])

def n_largest_factors(W_df,n_components,n=3):
    Factor_and_top_contributors={}
    for i in range(n_components):
       Factor_and_top_contributors[i] =W_df[i].nlargest(n)
    return Factor_and_top_contributors

In [54]:
Factorwise_top_contributors=n_largest_factors(W_df,n_components,5)
Factorwise_top_contributors_df=pd.DataFrame(Factorwise_top_contributors)

In [55]:
Factorwise_top_contributors_df

,0,1,2,3,4,5,6,7
Ticker,,,,,,,,
AXISBANK.NS,NaN,NaN,NaN,0.640441,NaN,NaN,NaN,NaN
BAJAJ-AUTO.NS,NaN,NaN,NaN,NaN,NaN,0.624094,NaN,NaN
BAJAJFINSV.NS,NaN,NaN,NaN,0.788852,NaN,NaN,NaN,NaN
BAJFINANCE.NS,NaN,NaN,NaN,0.806808,NaN,NaN,NaN,NaN
BHARTIARTL.NS,NaN,0.908306,NaN,NaN,NaN,NaN,NaN,NaN
BPCL.NS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.764835
BRITANNIA.NS,0.971461,NaN,0.632001,NaN,1.242327,NaN,NaN,NaN
CIPLA.NS,NaN,NaN,0.889261,NaN,NaN,NaN,NaN,NaN
COALINDIA.NS,NaN,0.831800,NaN,NaN,NaN,NaN,NaN,0.858864
